# 06_01 Writing Greek, one letter at a time

*📖 Book: sections 4.7 and 5.1.1*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → [model writes] → MCP tool
> ```
>
> You are here: **model writes**
>
> Project files this connects to:
>
> - `src/homer_ai/model.py`
> - `src/homer_ai/homer_model.pt`
>
> **This notebook:** We have a trained model. How do we turn its scores into written Greek, and why does the plainest way keep repeating itself?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How to turn the model's scores into one chosen letter
- Why we feed in only the last 24 letters
- How to build `write()` and write from three prompts
- Why greedy writing falls into a loop, shown with real numbers

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **repetition loop** | the model reads the same letters again and again, so it writes the same letters again and again | `και και και` |
| **context window** | the most recent letters the model can read at once; older ones fall out of view. Ours is 24 letters | `ids[-24:]` |

## 3. The idea

### ❓ Step 1: what does the trained model give me for the next letter?

The same as in [01_04](../01_foundations/01_04_our_model_as_a_black_box.ipynb): one row of 137 scores for each letter we give it (one score per token). Then softmax ([A_04](../A_toolbox/A_04_pytorch_in_this_repo.ipynb)) turns a row into chances that add up to 1. We read only the **last** row, because that's the guess for what comes after the last letter.

Two small things in the code. `model.eval()` switches off training-only tricks such as dropout ([03_03](../03_attention/03_03_hiding_the_future.ipynb)). Our `model.py` has none, so here it changes nothing, but it's the right habit. `torch.no_grad()` means "we only guess, we don't learn", which saves work.

👉 **Predict first:** after `ἄνδρα`, which character has the biggest chance? (The course map showed it.)

In [ ]:
import torch

from homer_ai.lessons import CHUNK, load_homer_model

# read-only: we only load the saved model, we never save
tokenizer, model = load_homer_model()  # yours, or else the ready-made one

In [ ]:
if model is not None:
    ids = tokenizer.encode("ἄνδρα")
    with torch.no_grad():
        scores = model(torch.tensor(ids))  # one row of scores for each of the 5 letters
    print("scores:", tuple(scores.shape), " (5 letters, 137 scores each)")

    # only the last row; dim=-1 means "across the 137 scores"
    chances = torch.softmax(scores[-1], dim=-1)
    print("chances add up to", round(chances.sum().item(), 4))
    best3 = torch.topk(chances, 3)  # the 3 biggest chances, and where they are
    for chance, token_id in zip(best3.values, best3.indices, strict=True):
        letter = tokenizer.decode([token_id.item()])
        print(f"  {letter!r:6} {chance.item():.2f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A space `' '` is the most likely next character (0.26), then `ι` (0.15) and `τ` (0.07). `!r` prints with quotes, so the invisible space shows up. Notice the chances are small even for the winner. The model is far from sure.

</details>

### ❓ Step 2: what do all the chances look like?

Here are the 10 biggest, as a picture. The orange bar is the one greedy would pick.

In [ ]:
if model is not None:
    import matplotlib.pyplot as plt

    from homer_ai.studio_style import ACCENT, INK, MUTED, studio_axes, studio_figure

    def show(ch):  # make invisible characters visible
        return {" ": "␣", "\n": "⏎"}.get(ch, ch)

    top = torch.topk(chances, 10)
    labels = [show(tokenizer.decode([i])) for i in top.indices.tolist()]
    values = top.values.tolist()

    fig = studio_figure("The next letter", "The model's chances after ἄνδρα", (10, 6))
    ax = studio_axes(fig)
    bars = ax.bar(labels, values, color=MUTED)
    bars[0].set_color(ACCENT)  # the greedy pick
    for i, v in enumerate(values):
        ax.text(i, v + 0.004, f"{v:.2f}", ha="center", color=INK, family="monospace")
    ax.tick_params(axis="x", labelsize=16, labelcolor=INK)
    ax.set_ylabel("chance")
    ax.grid(axis="x", visible=False)
    ax.set_axisbelow(True)  # grid lines behind the bars
    plt.show()
    print("the other 127 characters share:", round(1 - sum(values), 2))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

One bar stands out (`␣`, orange), but the next ones aren't far behind. And the 127 characters we can't see on the picture still hold nearly a quarter of all the chance (0.23). Keep that number in mind, because it's what [06_02](06_02_less_repetition.ipynb) plays with.

</details>

### ❓ Step 3: how do I pick one letter and add it?

With greedy picking. `argmax` gives the ID with the biggest chance, and `.item()` turns a one-number tensor into a plain number. Then we add it to the list of IDs. (Softmax never changes which number is biggest, so `argmax` of the scores would pick the same letter. We keep the chances step because 06_02 needs them.)

In [ ]:
if model is not None:
    best = chances.argmax().item()
    print("best ID:", best, "=", repr(tokenizer.decode([best])))
    ids = ids + [best]  # one new letter on the end
    print("text now:", repr(tokenizer.decode(ids)))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

ID 1 is the space, so the text grew from `'ἄνδρα'` to `'ἄνδρα '`. The space is hiding at the end, and `repr` puts quotes around it so you can see it.

</details>

### ❓ Step 4: why do we feed in only the last 24 letters?

The model trained on pieces of 24 letters, `chunk_size` ([02_04](../02_text_to_numbers/02_04_sliding_window.ipynb)), so 24 is the longest text it knows how to read. As the text grows we keep only the last 24, and older letters fall out of view, like a short-term memory. This is the **context window**.

Take the first 30 characters of Book 1:

```
all 30:   ἄνδρα μοι ἔννεπε, μοῦσα, πολύτ
kept 24:        μοι ἔννεπε, μοῦσα, πολύτ     (the model can't "see" ἄνδρα␣ here)
```

👉 **Predict first:** which 6 characters get dropped?

In [ ]:
from homer_ai.train_homer import BOOKS_DIR, load_books

if model is not None:
    line = load_books(BOOKS_DIR)[0][:30]
    all_ids = tokenizer.encode(line)
    print("all 30:  ", repr(line))
    print("kept 24: ", repr(tokenizer.decode(all_ids[-CHUNK:])))
    print("dropped: ", repr(tokenizer.decode(all_ids[:-CHUNK])))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`'ἄνδρα '` (5 letters and a space) is dropped, and `'μοι ἔννεπε, μοῦσα, πολύτ'` is kept. So once the text is longer than 24, the model has forgotten how the line began.

</details>

### ❓ Step 5: how do I put it all together?

Steps 1 to 4 in a loop. That's the whole of text generation, in a few lines. **Look at:** the 30 letters at the top. The model reads only the last 24 (orange), and the first 6 (red) are out of view. Follow the four steps along the row. The new letter (green) goes on the end, and the loop starts again.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    BAD,
    EDGE,
    GOOD,
    INK,
    MUTED,
    arrow,
    box,
    bracket,
    chips,
    diagram_axes,
    studio_figure,
    tag,
)

fig = studio_figure("The writing loop", "One new letter on every round", (10, 6.8))
ax = diagram_axes(fig)

# the text so far: 30 letters, and the new letter in green
text = "ἄνδρα μοι ἔννεπε, μοῦσα, πολύτ"
letters = [c if c != " " else "␣" for c in text]
colors = [BAD] * 6 + [EDGE] * 24 + [GOOD]
chips(ax, 0.4, 4.7, [*letters, "␣"], 0.28, 0.02, colors, fontsize=9)
bracket(ax, 0.4, 0.4 + 6 * 0.3 - 0.02, 5.1, "out of view", BAD, fontsize=9)
bracket(ax, 0.4 + 6 * 0.3, 0.4 + 30 * 0.3 - 0.02, 5.1, "the last 24 letters", ACCENT)
tag(ax, 9.83, 4.15, "new", GOOD, 9)

# the loop: four steps, then back to the text
steps = [
    (1.2, "1. read the\nlast 24 letters"),
    (3.7, "2. model:\n137 scores"),
    (6.2, "3. softmax:\nchances"),
    (8.7, "4. pick the best,\nadd it to the end"),
]
for x, label in steps:
    box(ax, x, 3.0, 1.9, 0.9, label, fontsize=9.5)
for x in (1.2, 3.7, 6.2):
    arrow(ax, (x + 0.97, 3.0), (x + 1.53, 3.0))
arrow(ax, (2.7, 4.45), (1.8, 3.5), color=ACCENT, width=2, curved=0.2)
arrow(ax, (8.7, 3.5), (9.83, 4.5), color=GOOD, width=2.2, curved=-0.25)

# the chances after ἄνδρα
arrow(ax, (6.2, 2.5), (6.2, 2.05), head=9)
for i, (letter, chance) in enumerate([("␣", 0.26), ("ι", 0.15), ("τ", 0.07)]):
    x = 5.6 + 0.6 * i
    ax.bar(x, chance * 3, width=0.45, bottom=0.9, color=ACCENT if i == 0 else MUTED)
    tag(ax, x, 0.65, letter, INK, 11, mono=True)
    tag(ax, x, 0.9 + chance * 3 + 0.15, f"{chance:.2f}", MUTED, 8, mono=True)
tag(ax, 8.7, 1.9, "greedy: the biggest\nchance wins", ACCENT, 9)
tag(ax, 1.2, 1.9, "older letters fall out\nof view, like a short\nmemory", BAD, 9)
plt.show()

In [ ]:
def write(model, tokenizer, prompt, n_new):
    ids = tokenizer.encode(prompt)
    with torch.no_grad():
        for _ in range(n_new):
            recent = torch.tensor(ids[-CHUNK:])  # the last 24 letters
            scores = model(recent)[-1]  # the guess after the last letter
            chances = torch.softmax(scores, dim=-1)
            ids.append(chances.argmax().item())  # greedy: the most likely letter
    return tokenizer.decode(ids).replace("\n", "⏎")  # show a line break as ⏎


if model is not None:
    print(write(model, tokenizer, "ἄνδρα", 60))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

You got `ἄνδρα τοι και και και κα κα κατον κατοι …`. The first words are real Greek: `τοι` ("to you") and `και` ("and"). But the order makes no sense, and it keeps coming back to `και`. Let's be fair to the model, though. It has 11,840 weights and has read 19,701 characters, one tiny book. GPT-2 small has about 124 million weights, roughly 10,000 times more, and the largest models are millions of times bigger.

</details>

### ❓ Your turn: which letter comes after `θεὰ`?

`write()` does one small thing over and over. Do that one thing yourself, once: ask the model for the most likely letter after `θεὰ` (goddess).

👉 **Predict first:** a letter, or a space?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
thea_ids = tokenizer.encode("θεὰ")

# 👉 do: get the model's scores after the last letter, pick the best one, decode it
# 💡 hint: model(...) wants a tensor and gives one row of scores per letter
next_letter = None

In [ ]:
from homer_ai.lessons import check

check(
    "the most likely letter after θεὰ",
    next_letter,
    write(model, tokenizer, "θεὰ", 1)[-1],
    coach="A number means you still have the ID: decode it, as a list of one, "
    "tokenizer.decode([...]). A wrong letter usually means the wrong row: the "
    "guess for the NEXT letter is the last row of scores, [-1].",
)

<details><summary>💡 Hint</summary>

Three lines of `write()` do it: `model(torch.tensor(...))[-1]` for the scores, `.argmax().item()` for the best ID, `tokenizer.decode([...])` for the letter. You can skip the softmax: the highest score is also the highest chance.

</details>

<details><summary>✅ Answer</summary>

```python
thea_ids = tokenizer.encode("θεὰ")

with torch.no_grad():
    thea_scores = model(torch.tensor(thea_ids))[-1]
next_letter = tokenizer.decode([thea_scores.argmax().item()])
```

With the ready-made model it's a space: `θεὰ` is a whole word, and the model has learned that words end. `write(model, tokenizer, "θεὰ", 1)` gives the same, because that's all `write()` does for one new letter.

</details>

### ❓ Does it write something different from other prompts?

Yes, but watch what happens. Two more prompts: `θεά` ("goddess", the Muse's title) and `Μοῦσα` ("Muse").

In [ ]:
if model is not None:
    for prompt in ["θεὰ", "Μοῦσα"]:
        print(f"{prompt!r:9} →", write(model, tokenizer, prompt, 60))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`θεὰ ἐπεν ἐπεν ἐπεν ἐπεν ἐπεν τον τον …` and `Μοῦσα έντοντοντοντοντοντον τοι τοι τοι τοι και και και και …`. A different start, and the same story: a word or two, then the same piece again and again. (`ἴδεν ἴδεν ἴδεν`, from the 5-line model in [05_03](../05_training_on_the_odyssey/05_03_memorising_vs_understanding.ipynb), is the same thing.)

</details>

### ❓ Why does it loop? Can I prove it?

Because greedy writing has no chance in it. The model is a fixed function, so the same 24 letters in always give the same letter out. The moment the last 24 letters are exactly the same as at an earlier step, everything after must repeat, for ever. That's a **repetition loop**.

We can catch it in the act. The code remembers every 24-letter window it has seen, and the step when it first saw it. When a window comes back, we stop and show the piece that repeats.

👉 **Predict first:** how many letters until it repeats: about 10, about 100 or about 1,000?

In [ ]:
def find_loop(prompt, max_steps=500):
    ids = tokenizer.encode(prompt)
    start = len(ids)
    seen = {}  # window → the step when we first saw it
    with torch.no_grad():
        for step in range(max_steps):
            window = tuple(ids[-CHUNK:])
            if window in seen:  # the same 24 letters as before!
                first = seen[window]
                return first, step, tokenizer.decode(ids[start + first : start + step])
            seen[window] = step
            ids.append(model(torch.tensor(window))[-1].argmax().item())
    return None


if model is not None:
    for prompt in ["ἄνδρα", "θεὰ", "Μοῦσα"]:
        first, again, piece = find_loop(prompt)
        print(f"{prompt!r:9} step {again} sees the window of step {first}:")
        print(f"          it repeats every {again - first} letters:", repr(piece))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Every prompt walks into a loop, after 74 to 222 letters. The repeating piece is 31 letters long each time, and it's the **same** piece for all three prompts (it only starts at a different place). So, different starts, one trap. After that point no letter is new. That's the problem the next notebook solves. If we sometimes pick a letter that isn't the best, the windows change and the loop breaks.

</details>

### ❓ Does the GPT of chapter 4 loop too?

Everything so far used the small one-head model of chapter 1. The GPT you built in [chapter 4](../04_building_the_model/04_05_the_full_model.ipynb) and measured in [05_05](../05_training_on_the_odyssey/05_05_validation.ipynb) has several heads, position rows and a context window of 64 letters instead of 24. `load_homer_gpt` loads it (yours, or the ready-made one). The helper `write` from `lessons.py` (imported as `write_both`) is the loop you just built, taught to read the right number of letters for either model. `top_k=1` means greedy.

👉 **Predict first:** the GPT is far better at guessing the next letter. Will greedy writing still loop?

In [ ]:
from homer_ai.lessons import load_homer_gpt
from homer_ai.lessons import write as write_both  # works for either model

gpt_tokenizer, gpt = load_homer_gpt()
print("the GPT reads", gpt.context_length, "letters at a time (the small model: 24)\n")
if model is not None:
    print("small:", write_both(model, tokenizer, "ἄνδρα", 100, top_k=1))
print("GPT:  ", write_both(gpt, gpt_tokenizer, "ἄνδρα", 100, top_k=1))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The small model says `και και και κα κα …`. The GPT writes `ἄνδρα πατρὸς ἀνέμενος ἀνθρώποισιν,⏎ἀλλλὰ δʼ ἀμφιάλοισι καὶ …`. Those are real Greek words with accents, the line breaks fall where Homer's lines end, and there's no early loop. It still doesn't mean anything. The words don't make a sentence, and some aren't words at all. But it does look like Homeric verse. The fair number is in [05_05](../05_training_on_the_odyssey/05_05_validation.ipynb): held-out loss 2.27.

</details>

### ❓ So is the loop gone?

No, it just comes later. The proof is the same as before: a greedy model is a fixed function, so once the last 64 letters repeat exactly, everything after repeats. Let's write 300 letters and look at the end.

In [ ]:
print(write_both(gpt, gpt_tokenizer, "ἄνδρα", 300, top_k=1)[-150:])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

At the end, the same line comes round again and again: `ἀλλλʼ ἀποιδήσαντα πατρὸς ἀνθρώποισι καὶ ἀτρεύσῃς,⏎`. In a test with three prompts, the GPT entered a loop after about 214 to 252 letters, with a 50-letter piece repeating. The small model took 74 to 222 letters, with a 31-letter piece. So a better model loops later, not never. The cure is still the dice of [06_02](06_02_less_repetition.ipynb).

</details>

## 4. Recap

- `write()` reads the last 24 letters → scores → chances → pick → add → repeat.
- Greedy always gives the same text, and the same 24 letters in always give the same letter out, so it ends in a loop (`και και και`).
- To break the loop we must sometimes pick a letter that isn't the most likely one.
- The GPT of chapter 4 (context 64) writes more like verse than the small model, but greedy writing still ends in a loop, only later.

**You can now:**

- get the model's most likely next letter for any text
- write the greedy loop: scores, pick, add, repeat
- show that greedy writing ends in a loop

**Next:** [06_02: Less repetition](06_02_less_repetition.ipynb): temperature and top-k.